In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18


# ── Dataset ───────────────────────────────────────────────────────────────────

# Run this once to confirm the true keypoints
def inspect_kp_ranges(root, n=50):
    xs, ys = [], []
    for file in sorted(os.listdir(root))[:n]:
        if not file.endswith("_pose.npz"):
            continue
        kp = np.load(os.path.join(root, file))["kp"][0]
        xs.append(kp[:, 0].max())
        ys.append(kp[:, 1].max())
    print(f"kp x max over {n} files: {max(xs):.1f}")
    print(f"kp y max over {n} files: {max(ys):.1f}")

# Debug showed kp x maxes at ~260, so source width is 320 (not 640).
# Uncomment to re-confirm: inspect_kp_ranges("./dataset")
KP_X_SRC = 320   # actual camera frame width your keypoints are in
KP_Y_SRC = 480   # actual camera frame height

HM_H = 256       # radar heatmap height
HM_W = 128       # radar heatmap width

VIS_THRESH = 0.3  # visibility is a float [0,1]; ignore low-confidence keypoints


class MMVRDataset(Dataset):
    def __init__(self, root, max_samples=200):
        self.samples = []
        for file in sorted(os.listdir(root)):
            if file.endswith("_radar.npz"):
                idx = file.replace("_radar.npz", "")
                self.samples.append(os.path.join(root, idx))
        self.samples = self.samples[:max_samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        base = self.samples[i]
        radar = np.load(base + "_radar.npz")

        hori = radar["hm_hori"].astype(np.float32)   # (256, 128) — always clean
        vert = radar["hm_vert"].astype(np.float32)   # (256, 128) — NaN in this section

        # FIX: handle NaNs before any further processing to prevent them from propagating through the pipeline
        hori = np.nan_to_num(hori, nan=0.0, posinf=0.0, neginf=0.0)
        vert = np.nan_to_num(vert, nan=0.0, posinf=0.0, neginf=0.0)

        # FIX: clip outliers to prevent them from dominating the normalization step
        hori = np.clip(hori, 0.0, np.percentile(hori, 99))
        vert = np.clip(vert, 0.0, np.percentile(vert, 99))

        hori = hori / (np.max(hori) + 1e-6)
        vert = vert / (np.max(vert) + 1e-6)

        radar_input = np.stack([hori, vert], axis=0)          # (2, H, W)
        radar_input = radar_input / (np.max(radar_input) + 1e-6)  # re-normalize after stacking

        # CHecking for NaNs after processing steps
        if np.isnan(radar_input).any():
            raise ValueError(f"NaN detected in radar input for sample {i}")

        pose = np.load(base + "_pose.npz")              # (2, H, W)
        kp   = pose["kp"][0].copy().astype(np.float32)  # (17, 3): x, y, visibility

        # Map keypoints from source image space to radar heatmap space, then clamp to valid pixel bounds
        kp[:, 0] = kp[:, 0] * HM_W / KP_X_SRC  # x -> [0, 128]
        kp[:, 1] = kp[:, 1] * HM_H / KP_Y_SRC  # y -> [0, 256]
        kp[:, 0] = np.clip(kp[:, 0], 0, HM_W - 1)
        kp[:, 1] = np.clip(kp[:, 1], 0, HM_H - 1)

        return (
            torch.tensor(radar_input, dtype=torch.float32),
            torch.tensor(kp,          dtype=torch.float32),
        )

# ── GT heatmap generation ─────────────────────────────────────────────────────

def make_heatmaps(kp_np, H=HM_H, W=HM_W, sigma=4):
    """kp_np: (17,3)  x in [0,W), y in [0,H), vis in [0,1]"""
    maps = np.zeros((17, H, W), dtype=np.float32)
    xx, yy = np.meshgrid(np.arange(W), np.arange(H))
    for j in range(17):
        x, y, v = kp_np[j]
        if v > VIS_THRESH:
            maps[j] = np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * sigma**2))
    return torch.tensor(maps, dtype=torch.float32)


# ── Model ─────────────────────────────────────────────────────────────────────

class SoftArgmax2D(nn.Module):
    def forward(self, x):
        B, C, H, W = x.shape

        # Subtract spatial max before softmax to prevent exp() overflow
        x = x - x.amax(dim=(2,3), keepdim=True)
        x_flat = F.softmax(x.view(B, C, -1), dim=-1)
        idx    = torch.arange(H * W, device=x.device).float()
        xs     = idx % W
        ys     = torch.div(idx, W, rounding_mode='floor')

        return torch.stack([
            torch.sum(x_flat * xs, dim=-1),
            torch.sum(x_flat * ys, dim=-1),
        ], dim=-1)   # (B, C, 2)


class PoseCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(2,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
        )
        self.res = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),
        )
        self.dec = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        )
        self.out         = nn.Conv2d(32, 17, 1)
        self.soft_argmax = SoftArgmax2D()

    def forward(self, x):
        x        = self.enc(x)
        x        = F.relu(x + self.res(x))
        x        = self.dec(x)
        heatmaps = self.out(x)
        coords   = self.soft_argmax(heatmaps)
        return heatmaps, coords


class ResNetPose(nn.Module):
    def __init__(self):
        super().__init__()
        self.net         = resnet18(weights=None)
        self.net.conv1   = nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False)
        self.net.fc      = nn.Linear(512, 34)

    def forward(self, x):
        return self.net(x).view(-1, 17, 2)


# ── Training ──────────────────────────────────────────────────────────────────

dataset = MMVRDataset("./dataset", max_samples=200)
loader  = DataLoader(dataset, batch_size=8, shuffle=True, drop_last=True)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Sanity check before training
x0, kp0 = dataset[0]
assert not torch.isnan(x0).any(),  "Still NaN in radar input!"
assert not torch.isnan(kp0).any(), "NaN in keypoints!"
print(f"Input OK  — range [{x0.min():.3f}, {x0.max():.3f}]")
print(f"kp x      — range [{kp0[:,0].min():.1f}, {kp0[:,0].max():.1f}]  (expect ~[0,{HM_W}])")
print(f"kp y      — range [{kp0[:,1].min():.1f}, {kp0[:,1].max():.1f}]  (expect ~[0,{HM_H}])")

# ── Model optimization ──────────────────────────────────────────────────────────

model     = PoseCNN().to(device)
baseline  = ResNetPose().to(device)
opt       = torch.optim.Adam(model.parameters(),    lr=1e-4)
# FIX: the ResNet baseline should optimize MSE on the raw keypoint coords, not heatmap.
mse_none = nn.MSELoss(reduction='none')


# ── Training Loop ──────────────────────────────────────────────────────────────

# Changing for number of training loops
epoch = 20

for epoch in range(epoch):
    model.train()
    baseline.train()

    total_loss = 0.0
    n_ok = 0

    for x, kp in loader:
        x  = x.to(device)
        kp = kp.to(device)

        gt_maps = torch.stack(
            [make_heatmaps(k.cpu().numpy()) for k in kp]
        ).to(device)

        pred_maps, _ = model(x)

        
        weight = (gt_maps > 0.5).float() * 49 + 1  # 50× weight on visible keypoints, 1× on background

        # FIX: divide by sum of weights, not pixel count, so the joint-center
        # error actually drives the reported loss.
        loss = (mse_none(pred_maps, gt_maps) * weight).sum() / weight.sum()

        if torch.isnan(loss):
            print(f"  [epoch {epoch}] NaN loss — skipping batch")
            continue

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        opt.step()


        total_loss   += loss.item()
        n_ok += 1

    # FIX: print once per EPOCH (was inside the batch loop, printing every iteration).

    print(f"Epoch {epoch:02d}  cnn_loss={total_loss / max(n_ok,1):.4f}  " 
          f"({n_ok}/{len(loader)} batches OK)")


# ── Metrics ───────────────────────────────────────────────────────────────────

def valid(gt, pred):
    """Return only the visible-keypoint rows AND the boolean mask used —
       so callers can index a per-joint confidence vector with the same mask.
    """
    mask = gt[:, 2] > VIS_THRESH
    return gt[mask, :2], pred[mask], mask


def mae(gt, pred):
    g, p, _ = valid(gt, pred)
    if len(g) == 0:
        return np.nan
    return np.mean(np.linalg.norm(g - p, axis=1))


def pck(gt, pred, frac=0.05):
    # FIX: must call valid() BEFORE referencing g (was a NameError).
    g, p, _ = valid(gt, pred)
    if len(g) == 0:
        return np.nan
    return np.mean(np.linalg.norm(g - p, axis=1) < frac * HM_H)


def oks(gt, pred):
    g, p, _ = valid(gt, pred)
    if len(g) == 0:
        return np.nan
    d_sq = np.sum((g - p)**2, axis=1)
    return np.mean(np.exp(-d_sq / (2 * (HM_H * HM_W) * 0.05**2)))


def precision_recall_f1(gt, pred, conf=None, conf_thresh=0.1, thresh_px=5.0):
    """Per-joint confidence defines abstention.
       TP : confident AND localized within thresh_px
       FP : confident AND beyond  thresh_px
       FN : visible GT but model abstained (low confidence) — independent of d
       If conf is None we treat every prediction as confident — which means
       Precision == Recall == PCK by construction (a property of a model
       with no abstention head, e.g. the ResNet baseline).
    """
    g, p, mask = valid(gt, pred)
    if len(g) == 0:
        return np.nan, np.nan, np.nan

    # FIX: was `if c is None` (c was undefined) and missed the else-branch
    # that masks `conf` down to the visible-only joints.
    if conf is None:
        c = np.ones(len(g), dtype=np.float32)
    else:
        c = np.asarray(conf)[mask]

    d  = np.linalg.norm(g - p, axis=1)
    pp = c > conf_thresh

    # FIX: FN = "visible GT but model abstained" — does NOT depend on d.
    # The previous version restricted FN to "would-have-been-TP misses",
    # which under-counts misses and inflates recall.
    tp = int(np.sum(pp & (d <  thresh_px)))
    fp = int(np.sum(pp & (d >= thresh_px)))
    fn = int(np.sum(~pp))

    precision = tp / (tp + fp + 1e-6)
    recall    = tp / (tp + fn + 1e-6)
    f1        = 2 * precision * recall / (precision + recall + 1e-6)
    return precision, recall, f1


# ── Evaluation ────────────────────────────────────────────────────────────────

model.eval()
baseline.eval()

x_s, kp_s = dataset[0]
x_in = x_s.unsqueeze(0).to(device)

with torch.no_grad():
    heat, pred_cnn = model(x_in)              # heat: (1, 17, H, W)  pred_cnn: (1, 17, 2)
    pred_resnet    = baseline(x_in)           # (1, 17, 2)

# FIX: drop the leading batch dim BEFORE handing the prediction to the metric
# funcs. valid() builds a (17,) mask from gt[:,2]; if pred still carries the
# batch dim it has shape (1, 17, 2) and `pred[mask]` tries to index axis 0
# (length 1) with a length-17 mask → IndexError.
pred_cnn_np    = pred_cnn[0].cpu().numpy()                     # (17, 2)
pred_resnet_np = pred_resnet[0].cpu().numpy()                  # (17, 2)
conf_cnn_np    = heat[0].amax(dim=(1, 2)).cpu().numpy()        # (17,)  per-joint heatmap peak
kp_b_np        = kp_s.numpy()                                  # (17, 3)

# FIX: pass the NUMPY arrays in the loop (was passing torch tensors that
# still had the batch dim, which is what triggered the IndexError).
# Also pass per-joint confidence for the CNN so Precision/Recall can diverge.
for name, pred, conf in [
    ("CNN (ours)",         pred_cnn_np,    conf_cnn_np),
    ("ResNet-18 baseline", pred_resnet_np, None),
]:
    print(f"\n=== {name} ===")
    print(f"  MAE       : {mae(kp_b_np, pred):.3f} px")
    print(f"  PCK@5%    : {pck(kp_b_np, pred):.3f}")
    print(f"  OKS       : {oks(kp_b_np, pred):.3f}")
    p, r, f = precision_recall_f1(kp_b_np, pred, conf=conf)
    print(f"  Precision : {p:.3f}")
    print(f"  Recall    : {r:.3f}")
    print(f"  F1        : {f:.3f}")


Using device: cpu
Input OK  — range [0.000, 1.000]
kp x      — range [21.0, 92.7]  (expect ~[0,128])
kp y      — range [3.1, 252.9]  (expect ~[0,256])
Epoch 00  cnn_loss=0.1285  (25/25 batches OK)
Epoch 01  cnn_loss=0.0726  (25/25 batches OK)
Epoch 02  cnn_loss=0.0566  (25/25 batches OK)
Epoch 03  cnn_loss=0.0460  (25/25 batches OK)
Epoch 04  cnn_loss=0.0379  (25/25 batches OK)
Epoch 05  cnn_loss=0.0330  (25/25 batches OK)
Epoch 06  cnn_loss=0.0298  (25/25 batches OK)
Epoch 07  cnn_loss=0.0278  (25/25 batches OK)
Epoch 08  cnn_loss=0.0264  (25/25 batches OK)
Epoch 09  cnn_loss=0.0254  (25/25 batches OK)
Epoch 10  cnn_loss=0.0244  (25/25 batches OK)
Epoch 11  cnn_loss=0.0237  (25/25 batches OK)
Epoch 12  cnn_loss=0.0231  (25/25 batches OK)
Epoch 13  cnn_loss=0.0226  (25/25 batches OK)
Epoch 14  cnn_loss=0.0223  (25/25 batches OK)
Epoch 15  cnn_loss=0.0218  (25/25 batches OK)
Epoch 16  cnn_loss=0.0217  (25/25 batches OK)
Epoch 17  cnn_loss=0.0214  (25/25 batches OK)
Epoch 18  cnn_loss=0.